# SQL GROUP BY, JOINs and CTEs
**Goal:** summarize Nkwen Traders sales by group, combine the sales table with the branch table, and write your first CTE.

**What to submit:** the GitHub commit link to this notebook, and in the text box your answers (tables or numbers) with the SQL for Q1 to Q5 and your two-sentence explanation for Q4(b).

In [ ]:
import sqlite3
import pandas as pd

SALES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_ch2_sales.csv"
BRANCHES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_branches.csv"

# 1. Load the two CSV files into pandas (Colab downloads them, not your phone or laptop)
sales = pd.read_csv(SALES_URL)
branches = pd.read_csv(BRANCHES_URL)

# 2. Create a small SQLite database file and copy both DataFrames into it as tables
conn = sqlite3.connect("nkwen.db")
sales.to_sql("sales", conn, if_exists="replace", index=False)
branches.to_sql("branches", conn, if_exists="replace", index=False)

# 3. A helper: run a SQL query and get the result back as a DataFrame
def run_sql(query):
    return pd.read_sql_query(query, conn)

print("SQLite version:", sqlite3.sqlite_version)
print("Tables:", run_sql("SELECT name FROM sqlite_master WHERE type = 'table'")["name"].tolist())

### Self-check 1: are the tables loaded?

In [ ]:
assert run_sql("SELECT COUNT(*) AS n FROM sales")["n"][0] == len(sales), "Re-run the setup cell."
assert run_sql("SELECT COUNT(*) AS n FROM branches")["n"][0] == len(branches), "Re-run the setup cell."
TOTAL_REVENUE = run_sql("SELECT SUM(TotalSale_FCFA) AS t FROM sales")["t"][0]
TOTAL_ROWS = len(sales)
print("Success: both tables are loaded. The checks below compare your results with these totals.")

## Step 1: GROUP BY
`GROUP BY` makes one row per group, and the aggregate (`SUM`, `COUNT`, `AVG`, `MIN`, `MAX`) is computed inside each group.

```sql
SELECT PaymentMethod, COUNT(*) AS transactions
FROM sales
GROUP BY PaymentMethod
ORDER BY transactions DESC;
```
**Rule:** every column in `SELECT` that is not inside an aggregate must also be in `GROUP BY`.
Most databases give an error if you break this rule. **SQLite does not**: it silently picks a value from a random row. So check your `GROUP BY` yourself.

In [ ]:
run_sql("""
    SELECT PaymentMethod, COUNT(*) AS transactions
    FROM sales
    GROUP BY PaymentMethod
    ORDER BY transactions DESC
""")

### Q1. Revenue and number of transactions per branch, highest revenue first
Columns: `Branch, revenue, transactions`.

In [ ]:
q1_sql = """
-- TODO
"""
r1 = run_sql(q1_sql)
r1

### Self-check 2: do the parts add up to the whole?

In [ ]:
assert {"revenue", "transactions"}.issubset(r1.columns), "Name your columns with AS revenue and AS transactions."
assert r1["revenue"].sum() == TOTAL_REVENUE, "Branch revenues should add up to the total revenue of the sales table."
assert r1["transactions"].sum() == TOTAL_ROWS, "Branch transactions should add up to the number of rows in sales."
assert r1["revenue"].is_monotonic_decreasing, "Sort with ORDER BY revenue DESC."
print("Success: your branch totals add up to the company totals.")

### Q2. Average transaction value per category, rounded to whole FCFA, highest first
Hint: `ROUND(AVG(TotalSale_FCFA)) AS avg_transaction_value`.

In [ ]:
q2_sql = """
-- TODO
"""
r2 = run_sql(q2_sql)
r2

## Step 2: WHERE or HAVING?
- `WHERE` filters **rows before** grouping (for example only Wholesale sales).
- `HAVING` filters **groups after** grouping (for example only products whose total is above a value).

### Q3. Which products made more than 2,000,000 FCFA in total (all branches together)? Highest first.
Columns: `Product, revenue`.

In [ ]:
q3_sql = """
-- TODO
"""
r3 = run_sql(q3_sql)
r3

## Step 3: JOINs
The `branches` table has each branch's Region, City and Manager. A JOIN puts the two tables side by side, matching on the branch name:

```sql
SELECT s.TransactionID, s.Branch, b.Region, s.TotalSale_FCFA
FROM sales AS s
JOIN branches AS b ON s.Branch = b.Branch
LIMIT 5;
```
- `JOIN` (same as `INNER JOIN`) keeps only rows that match in **both** tables.
- `LEFT JOIN` keeps **every row of the left table**, and fills the right side with NULL when there is no match.
- `COUNT(*)` counts rows, including a row filled with NULLs. `COUNT(s.TransactionID)` counts only real sales.
- `COALESCE(SUM(x), 0)` shows 0 instead of NULL.
- Other databases also have `RIGHT JOIN` and `FULL OUTER JOIN`. Only newer SQLite versions (3.39 and later) have them, so here we put the table we want to keep on the **left** and use `LEFT JOIN`.

### Q4(a). Revenue by Region (inner join), highest first
Columns: `Region, revenue`.

In [ ]:
q4a_sql = """
-- TODO
"""
r4a = run_sql(q4a_sql)
r4a

### Q4(b). Every branch in the branch table, with City, number of transactions and revenue (0 when there are no sales)
Columns: `Branch, City, transactions, revenue`. Start `FROM branches AS b LEFT JOIN sales AS s ...`.

In [ ]:
q4b_sql = """
-- TODO
"""
r4b = run_sql(q4b_sql)
r4b

In [ ]:
# Self-check 3 (Q3 and Q4)
assert "HAVING" in q3_sql.upper(), "Q3: filter the groups with HAVING, not WHERE."
assert (r3["revenue"] > 2_000_000).all(), "Q3: every product should be above 2,000,000 FCFA. Name the total AS revenue."
assert r4a["revenue"].sum() == TOTAL_REVENUE, "Region revenues should add up to the total revenue. Name the total AS revenue."
assert len(r4b) == len(branches), "Q4(b) should have one row for every branch in the branches table. Did you start FROM branches and use LEFT JOIN?"
assert r4b["revenue"].notna().all(), "Show 0 instead of an empty revenue: use COALESCE(SUM(...), 0)."
print("Success: Q3 keeps only big products, regions add up to the total, and every branch from the branch table is listed.")

**Q4(b) explanation (TODO, 2 sentences for the operations manager):** what does this table show that an inner join would hide?

## Step 4: Common Table Expressions (CTEs)
A CTE is a named, temporary result you build first and then query, like a helper table. It keeps long queries readable:

```sql
WITH monthly AS (
    SELECT Branch,
           strftime('%Y-%m', Date) AS month,   -- '2026-03' (text)
           SUM(TotalSale_FCFA) AS revenue
    FROM sales
    GROUP BY Branch, month
)
SELECT * FROM monthly ORDER BY Branch, month;
```

### Q5. Using a CTE, find each branch's best month (the month with its highest revenue) and that revenue
Columns: `Branch, month, revenue`. Hint: build `monthly` as above, then a second step that finds the maximum revenue per branch, and join the two on Branch and revenue.

In [ ]:
q5_sql = """
-- TODO
"""
r5 = run_sql(q5_sql)
r5

In [ ]:
# Self-check 4
assert q5_sql.strip().upper().startswith("WITH"), "Q5 should start with WITH (a CTE)."
assert len(r5) == r5["Branch"].nunique(), "Each branch should appear once."
assert len(r5) == sales["Branch"].nunique(), "You should have one row for each branch that has sales."
print("Success: one best month for each branch.")

### Before you submit
- [ ] **Runtime > Restart session and run all**: every cell runs from top to bottom without an error.
- [ ] Every self-check cell printed its success message.
- [ ] Your answers are written in Markdown cells in your own words, not only as code output.
- [ ] Save to GitHub: **File > Save a copy in GitHub**, choose your repository, write a clear commit message, and click OK.
- [ ] Open the notebook on GitHub, check that the outputs show, and copy the commit link for your submission.